# CoCa Sweep Analysis
Automatically loads all run results from `checkpoints/` and compares models across metrics.

In [ ]:
import os
import json
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 12, "figure.dpi": 120})

CHECKPOINT_DIR = "checkpoints"

# ── Load all runs ──
runs = []
for run_dir in sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "*"))):
    config_path = os.path.join(run_dir, "config.json")
    summary_path = os.path.join(run_dir, "summary.json")
    metrics_path = os.path.join(run_dir, "metrics.csv")
    if not os.path.isfile(metrics_path):
        continue
    
    config = json.load(open(config_path)) if os.path.isfile(config_path) else {}
    summary = json.load(open(summary_path)) if os.path.isfile(summary_path) else {}
    metrics = pd.read_csv(metrics_path)
    
    run_name = os.path.basename(run_dir)
    runs.append({
        "name": run_name,
        "config": config,
        "summary": summary,
        "metrics": metrics,
        "ts_arch": config.get("model", {}).get("ts_arch", "?"),
        "decoder_arch": config.get("model", {}).get("decoder_arch", "?"),
        "text_source": config.get("data", {}).get("text_source", "?"),
        "lr": config.get("training", {}).get("learning_rate", 0),
        "temperature": config.get("model", {}).get("temperature", 0),
    })

print(f"Loaded {len(runs)} runs:")
for r in runs:
    epochs = len(r['metrics'])
    best = r['summary'].get('best_val_loss', '?')
    print(f"  {r['name']:40s}  epochs={epochs:2d}  best_val={best}")

## 1. Summary Table — Best Metrics per Run

In [ ]:
rows = []
for r in runs:
    m = r["metrics"]
    best_epoch = m["val_loss"].idxmin()
    best_row = m.iloc[best_epoch]
    rows.append({
        "Run": r["name"],
        "TS Arch": r["ts_arch"],
        "Decoder": r["decoder_arch"],
        "Text Source": r["text_source"],
        "LR": r["lr"],
        "Temp": r["temperature"],
        "Best Epoch": int(best_row["epoch"]),
        "Val Loss": best_row["val_loss"],
        "Val Cap": best_row["val_caption"],
        "Val Con": best_row["val_contrastive"],
        "ECG→Text R@1": best_row.get("val_ecg2text_R@1", 0),
        "ECG→Text R@5": best_row.get("val_ecg2text_R@5", 0),
        "Text→ECG R@1": best_row.get("val_text2ecg_R@1", 0),
        "Test Loss": r["summary"].get("test_loss", None),
    })

summary_df = pd.DataFrame(rows).sort_values("Val Loss")
summary_df.style.format(precision=4).highlight_min(
    subset=["Val Loss", "Val Cap", "Val Con"], color="#d4edda"
).highlight_max(
    subset=["ECG→Text R@1", "ECG→Text R@5", "Text→ECG R@1"], color="#d4edda"
)

## 2. Training Curves — Validation Loss

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for r in runs:
    m = r["metrics"]
    axes[0].plot(m["epoch"], m["val_loss"], label=r["name"], marker=".", markersize=3)
    axes[1].plot(m["epoch"], m["val_caption"], label=r["name"], marker=".", markersize=3)
    axes[2].plot(m["epoch"], m["val_contrastive"], label=r["name"], marker=".", markersize=3)

for ax, title in zip(axes, ["Total Val Loss", "Caption Loss", "Contrastive Loss"]):
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

axes[0].legend(bbox_to_anchor=(0, -0.35), loc="upper left", ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig("analysis_loss_curves.png", bbox_inches="tight")
plt.show()


## 3. Retrieval Performance (R@1, R@5)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for r in runs:
    m = r["metrics"]
    if "val_ecg2text_R@1" in m.columns:
        axes[0].plot(m["epoch"], m["val_ecg2text_R@1"], label=r["name"], marker=".", markersize=3)
        axes[1].plot(m["epoch"], m["val_ecg2text_R@5"], label=r["name"], marker=".", markersize=3)

for ax, title in zip(axes, ["ECG→Text R@1", "ECG→Text R@5"]):
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Recall")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

axes[0].legend(bbox_to_anchor=(0, -0.35), loc="upper left", ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig("analysis_retrieval.png", bbox_inches="tight")
plt.show()

## 4. Bar Chart — Best R@1 per Model

In [ ]:
if len(summary_df) > 0 and "ECG→Text R@1" in summary_df.columns:
    df_sorted = summary_df.sort_values("ECG→Text R@1", ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, max(4, len(df_sorted) * 0.5)))
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(df_sorted)))
    bars = ax.barh(df_sorted["Run"], df_sorted["ECG→Text R@1"], color=colors)
    ax.set_xlabel("ECG→Text Recall@1")
    ax.set_title("Best Retrieval R@1 per Experiment")
    ax.grid(True, axis="x", alpha=0.3)
    
    for bar, val in zip(bars, df_sorted["ECG→Text R@1"]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f"{val:.3f}", va="center", fontsize=9)
    
    plt.tight_layout()
    plt.savefig("analysis_r1_comparison.png", bbox_inches="tight")
    plt.show()
else:
    print("No retrieval data available yet.")

## 5. Grouped Comparison — Decoder Architecture

In [ ]:
if len(summary_df) > 1:
    metrics_to_compare = ["Val Loss", "ECG→Text R@1", "ECG→Text R@5"]
    available = [c for c in metrics_to_compare if c in summary_df.columns]
    
    if available:
        grouped = summary_df.groupby("Decoder")[available].mean()
        fig, axes = plt.subplots(1, len(available), figsize=(5 * len(available), 5))
        if len(available) == 1:
            axes = [axes]
        
        for ax, col in zip(axes, available):
            grouped[col].plot(kind="bar", ax=ax, color=plt.cm.Set2.colors)
            ax.set_title(f"Mean {col} by Decoder")
            ax.set_ylabel(col)
            ax.tick_params(axis="x", rotation=45)
            ax.grid(True, axis="y", alpha=0.3)
        
        plt.tight_layout()
        plt.savefig("analysis_decoder_comparison.png", bbox_inches="tight")
        plt.show()
else:
    print("Need multiple runs to compare.")

## 6. Hyperparameter Impact — LR & Temperature

In [ ]:
if len(summary_df) > 2:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # LR vs Val Loss
    for decoder in summary_df["Decoder"].unique():
        subset = summary_df[summary_df["Decoder"] == decoder]
        axes[0].scatter(subset["LR"], subset["Val Loss"], label=decoder, s=80)
    axes[0].set_xscale("log")
    axes[0].set_xlabel("Learning Rate")
    axes[0].set_ylabel("Best Val Loss")
    axes[0].set_title("LR vs Val Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Temperature vs R@1
    if "ECG→Text R@1" in summary_df.columns:
        for decoder in summary_df["Decoder"].unique():
            subset = summary_df[summary_df["Decoder"] == decoder]
            axes[1].scatter(subset["Temp"], subset["ECG→Text R@1"], label=decoder, s=80)
        axes[1].set_xlabel("Temperature")
        axes[1].set_ylabel("ECG→Text R@1")
        axes[1].set_title("Temperature vs Retrieval R@1")
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig("analysis_hyperparam_impact.png", bbox_inches="tight")
    plt.show()
else:
    print("Need more runs to analyze hyperparameter impact.")

## 7. Export Results

In [ ]:
summary_df.to_csv("sweep_results_summary.csv", index=False)
print(f"Saved sweep_results_summary.csv with {len(summary_df)} runs.")
print("\nTop 3 by Val Loss:")
display(summary_df.head(3)[["Run", "Decoder", "Val Loss", "ECG→Text R@1", "ECG→Text R@5"]])